# Actividad 12. Interpretabilidad, equidad y estabilidad

Un modelo útil no solo debe acertar: también debe ser comprensible, comportarse
de forma razonable en distintos grupos y mantener su desempeño en el tiempo.
Esta actividad responde tres preguntas:

1. ¿Qué variables aportan más y qué pasa si se eliminan?
2. ¿El desempeño es parecido entre regiones, niveles de PIB y tipos de desastre?
3. ¿Se mantiene la calidad entre 2024 y 2025?

## 1. Configuración

In [1]:
import json
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados" / "ml"
GRAFICAS = RAIZ / "resultados" / "graficas" / "ml"
TABLAS = RAIZ / "resultados" / "tablas"
for carpeta in (RESULTADOS, GRAFICAS, TABLAS):
    carpeta.mkdir(parents=True, exist_ok=True)

PALETA = {
    "purpura": "#4C1D95",
    "purpura_claro": "#8B5CF6",
    "rosa": "#FA8A9A",
    "amarillo": "#FFB71B",
    "rojo": "#EE3E29",
    "coral": "#FF8A5F",
    "azul": "#1B4576",
    "gris": "#7A7191",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 150,
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.22,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
    }
)


def guardar(fig, nombre):
    ruta = GRAFICAS / nombre
    fig.savefig(ruta, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Figura guardada: {ruta.relative_to(RAIZ)}")

In [2]:
ablacion = pd.read_csv(RESULTADOS / "ablacion_xgb.csv")
equidad = pd.read_csv(RESULTADOS / "equidad_modelos.csv")
estabilidad = pd.read_csv(RESULTADOS / "estabilidad_temporal.csv")
shap = pd.read_csv(RESULTADOS / "shap_xgb_clasificacion.csv")

print(f"Variantes de ablación: {len(ablacion)}")
print(f"Grupos evaluados: {equidad.groupby(['dimension', 'grupo']).ngroups}")
print(f"Años de prueba: {', '.join(str(a) for a in sorted(estabilidad['anio'].unique()))}")

Variantes de ablación: 5
Grupos evaluados: 17
Años de prueba: 2024, 2025


## 2. Ablación de variables

Se reentrena XGBoost quitando un grupo de variables a la vez. La caída de
ROC-AUC indica cuánto aporta ese grupo. Si al quitarlo el desempeño no baja,
la variable no es imprescindible.

In [3]:
display(ablacion.round(4))
ablacion.to_csv(TABLAS / "actividad12_ablacion.csv", index=False)

fig, eje = plt.subplots(figsize=(8.8, 4.2))
datos = ablacion.sort_values("caida_roc_auc")
colores = [PALETA["rojo"] if valor > 0.01 else PALETA["purpura_claro"] for valor in datos["caida_roc_auc"]]
eje.barh(datos["variante"], datos["caida_roc_auc"], color=colores)
eje.set_xlabel("Caída de ROC-AUC al quitar el grupo")
eje.set_title("Ablación de variables de XGBoost")
for i, valor in enumerate(datos["caida_roc_auc"]):
    eje.text(valor + 0.0005, i, f"{valor:.4f}", va="center", fontsize=8)
guardar(fig, "actividad12_ablacion.png")

,variante,n_variables,roc_auc,pr_auc,f1,mejor_iteracion,caida_roc_auc
0,Modelo completo,268,0.8267,0.7710,0.7103,42,0.0000
1,Sin PIB per cápita,266,0.8090,0.7473,0.6954,72,0.0177
2,Sin magnitud ni escala,259,0.8215,0.7760,0.7020,55,0.0052
3,"Sin país, región ni subregión",66,0.8169,0.7687,0.6849,43,0.0098
4,Sin estacionalidad,265,0.8194,0.7679,0.6951,92,0.0073


Figura guardada: resultados\graficas\ml\actividad12_ablacion.png


## 3. Contribución por familia de variables

El SHAP individual se agrupa por familia para obtener una lectura más
estable: economía, demografía, tiempo, magnitud, ubicación y tipo de evento.

In [4]:
def familia(variable):
    if variable.startswith("cat__"):
        return variable.split("__")[1]
    if variable.startswith("num__"):
        return variable[5:]
    return variable

shap["familia"] = shap["variable"].map(familia)
por_familia = (
    shap.groupby("familia", as_index=False)["shap_medio_absoluto"]
    .sum()
    .sort_values("shap_medio_absoluto", ascending=False)
)
display(por_familia.round(4))
por_familia.to_csv(TABLAS / "actividad12_shap_familias.csv", index=False)

fig, eje = plt.subplots(figsize=(8.5, 4.2))
eje.bar(por_familia["familia"], por_familia["shap_medio_absoluto"], color=PALETA["purpura_claro"])
eje.set_ylabel("Contribución SHAP acumulada")
eje.set_title("Peso de cada familia de variables")
eje.tick_params(axis="x", rotation=25)
guardar(fig, "actividad12_shap_familias.png")

,familia,shap_medio_absoluto
11,pib_pc_log,0.4022
2,duracion_log,0.3821
16,subtipo,0.3190
17,tipo,0.2364
12,poblacion_log,0.2009
13,region,0.1963
1,duracion_faltante,0.1495
4,iso,0.1354
0,densidad_log,0.1228
6,magnitud_z,0.1174


Figura guardada: resultados\graficas\ml\actividad12_shap_familias.png


## 4. Equidad entre grupos

Se comparan las ROC-AUC de prueba por región, cuartil de PIB y tipo de
desastre. Un grupo con muy pocos eventos se excluye para evitar conclusiones
sobre muestras pequeñas.

In [5]:
dimensiones = [
    ("region", "Región"),
    ("pib_cuartil", "Cuartil de PIB"),
    ("tipo", "Tipo de desastre"),
]
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.6))
for eje, (dimension, titulo) in zip(ejes, dimensiones):
    sub = equidad[equidad["dimension"] == dimension]
    pivote = sub.pivot(index="grupo", columns="modelo", values="roc_auc").sort_values("xgboost")
    pivote.plot(kind="barh", ax=eje, color=[PALETA["rojo"], PALETA["purpura"]], width=0.75)
    eje.set_title(titulo)
    eje.set_xlabel("ROC-AUC")
    eje.set_xlim(0.5, 1.0)
    eje.set_ylabel("")
    eje.legend(frameon=False, fontsize=8)
guardar(fig, "actividad12_equidad.png")

brechas = (
    equidad.groupby(["dimension", "modelo"])["roc_auc"]
    .agg(["min", "max", "mean", "count"])
    .reset_index()
)
brechas["brecha_max_min"] = brechas["max"] - brechas["min"]
display(brechas.round(4))
brechas.to_csv(TABLAS / "actividad12_equidad_brechas.csv", index=False)

for dimension, titulo in dimensiones:
    sub = equidad[equidad["dimension"] == dimension]
    peor = sub.loc[sub["roc_auc"].idxmin()]
    print(f"{titulo}: peor grupo = {peor['grupo']} ({peor['modelo']}, ROC-AUC {peor['roc_auc']:.3f}, n={int(peor['n'])})")

Figura guardada: resultados\graficas\ml\actividad12_equidad.png


,dimension,modelo,min,max,mean,count,brecha_max_min
0,pib_cuartil,red_neuronal,0.7134,0.9046,0.7958,4,0.1913
1,pib_cuartil,xgboost,0.7225,0.8980,0.7999,4,0.1755
2,region,red_neuronal,0.6389,0.9329,0.7719,5,0.2941
3,region,xgboost,0.6389,0.9242,0.7707,5,0.2853
4,tipo,red_neuronal,0.5983,1.0000,0.7684,8,0.4017
5,tipo,xgboost,0.5680,0.9333,0.7450,8,0.3653


Región: peor grupo = Oceania (red_neuronal, ROC-AUC 0.639, n=15)
Cuartil de PIB: peor grupo = PIB alto (red_neuronal, ROC-AUC 0.713, n=203)
Tipo de desastre: peor grupo = Epidemic (xgboost, ROC-AUC 0.568, n=26)


## 5. Estabilidad temporal

La prueba cubre dos años. Si la ROC-AUC cambia mucho entre 2024 y 2025, la
estimación es sensible al año y conviene comunicarlo.

In [6]:
display(estabilidad.round(4))
estabilidad.to_csv(TABLAS / "actividad12_estabilidad.csv", index=False)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for eje, metrica, titulo in [
    (ejes[0], "roc_auc", "ROC-AUC por año"),
    (ejes[1], "f1", "F1 por año"),
]:
    for modelo, color, etiqueta in [
        ("red_neuronal", PALETA["rojo"], "Red neuronal"),
        ("xgboost", PALETA["purpura"], "XGBoost"),
    ]:
        sub = estabilidad[estabilidad["modelo"] == modelo]
        eje.plot(sub["anio"], sub[metrica], marker="o", linewidth=2, color=color, label=etiqueta)
    eje.set_xticks(sorted(estabilidad["anio"].unique()))
    eje.set_title(titulo)
    eje.set_ylim(0.55, 0.9)
    eje.legend(frameon=False)
guardar(fig, "actividad12_estabilidad.png")

,anio,modelo,n,positivos,prevalencia,roc_auc,f1,precision,recall
0,2024,red_neuronal,424,179,0.4222,0.8280,0.7245,0.6667,0.7933
1,2024,xgboost,424,179,0.4222,0.8334,0.7387,0.6712,0.8212
2,2025,red_neuronal,389,142,0.3650,0.8078,0.6807,0.5947,0.7958
3,2025,xgboost,389,142,0.3650,0.8195,0.6788,0.5957,0.7887


Figura guardada: resultados\graficas\ml\actividad12_estabilidad.png

## Conclusión

* El PIB per cápita es el grupo que más aporta: al quitarlo, la ROC-AUC cae
  alrededor de **0.018**. La magnitud y la estacionalidad aportan menos.
* El desempeño es razonablemente estable entre 2024 y 2025; XGBoost mantiene
  una ROC-AUC cercana a **0.82** en ambos años.
* Hay diferencias entre grupos: las regiones con menos eventos y los tipos de
  desastre poco frecuentes presentan mayor incertidumbre. La aplicación
  advierte cuando un país no tiene historial suficiente.
* La interpretabilidad se comunica como **asociación**, no como causalidad.